In [1]:
!pip install transformers datasets accelerate -q

In [5]:
from datasets import Dataset
data = {
    "text": [
        "Artificial intelligence is the new electricity.",
        "Machine learning transforms data into intelligence.",
        "Deep learning enables machines to see and hear.",
        "AI will reshape every industry in the coming decade.",
        "The future belongs to those who understand algorithms.",
        "Neural networks mimic the structure of the brain.",
        "Data is the fuel that powers intelligent systems.",
        "Innovation begins with curiosity and computation.",
        "Automation enhances human productivity.",
        "AI is not magic, it is mathematics at scale."
    ]
}

dataset = Dataset.from_dict(data)
dataset

Dataset({
    features: ['text'],
    num_rows: 10
})

In [6]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
tokenizer.pad_token = tokenizer.eos_token

In [10]:
def tokenize_function(example):
  return tokenizer(
      example['text'],
      truncation=True,
      padding='max_length',
      max_length=64
  )
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [11]:
tokenized_dataset

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 10
})

In [12]:
tokenized_dataset = tokenized_dataset.remove_columns(['text'])

In [13]:
tokenized_dataset.set_format('torch')

In [14]:
from transformers import Trainer, TrainingArguments

In [16]:
training_args = TrainingArguments(
    output_dir='./gpt2-finetuned-ai',
    num_train_epochs=20,
    per_device_train_batch_size=2,
    save_steps=10,
    save_total_limit=2,
    logging_steps=5,
    learning_rate=5e-5,
    weight_decay=-0.01,
    fp16=True
)

In [18]:
trainer = Trainer(
    model=model,
    args= training_args,
    train_dataset = tokenized_dataset
)

In [19]:
from transformers import DataCollatorForLanguageModeling

In [20]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

In [21]:
trainer = Trainer(
    model=model,
    args= training_args,
    train_dataset = tokenized_dataset,
    data_collator = data_collator
)

In [22]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.271369
10,3.992957
15,2.689182
20,1.947225
25,1.527202
30,0.972090
35,0.545436
40,0.608462
45,0.465736
50,0.493354


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=1.0366582190990448, metrics={'train_runtime': 210.8434, 'train_samples_per_second': 0.949, 'train_steps_per_second': 0.474, 'total_flos': 6532300800000.0, 'train_loss': 1.0366582190990448, 'epoch': 20.0})

In [23]:
trainer.save_model('./gpt-finetuned-ai')
tokenizer.save_pretrained('./gpt-finetuned-ai')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt-finetuned-ai/tokenizer_config.json',
 './gpt-finetuned-ai/tokenizer.json')

In [24]:
from transformers import pipeline

generator = pipeline(
    'text-generation',
    model='./gpt-finetuned-ai',
    tokenizer='./gpt-finetuned-ai'
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [25]:
prompt = 'Artificial intelligence'

output = generator(
    prompt,
    max_length=15,
    num_return_sequences=1,
    temperature=0.2,
    top_k=500,
    top_p=0.2
)

for i, seq in enumerate(output):
    print(f"Generated {i+1}: {seq['generated_text']}\n")

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'top_p', 'temperature', 'max_length', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=15) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated 1: Artificial intelligence is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electricity. It is the new electri